# Train the CiteAudit classifier on a Colab GPU

Trains `scripts/train_classifier.py` (BiomedBERT, evidence packed into 512-token windows) on the
top-20 evidence from the paper's BM25+MonoT5 retrieval, picks the epoch with the best dev
macro-F1, and scores it on the test split with both the paper's evidence and our reranker's.

**Before running:** Runtime → Change runtime type → **T4 GPU** (or better). Then Runtime → Run all.
On a T4 this takes roughly 20–30 minutes. Every input it needs is committed to the repository,
so nothing has to be uploaded.

The number to beat is the paper's top-20 test **macro-F1 0.52** (the released MultiVerS
checkpoint gives 0.524 on the same evidence; dev 0.684).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/SapnilPatel/CiteAudit.git
%cd CiteAudit
!git log -1 --oneline
!pip -q install "transformers>=4.44" "tokenizers>=0.19"

## Train

`--amp` turns on fp16 mixed precision, about twice as fast on a T4. Progress prints once per epoch.

In [ ]:
!python scripts/train_classifier.py \
    --train data_train/target/paper_20_train --dev data_train/target/paper_20 \
    --out checkpoints/classifier --pred-dir results/classification/classifier \
    --predict test=data_train/target/paper_20_test --predict test=data_train/target/ours_20_test \
    --amp --eval-batch 16 2>&1 | grep -vE "UNEXPECTED|MISSING|LOAD REPORT|^Key|^-+|^Notes|^- |Loading weights"

## Score

Same scorer as the MultiVerS runs, so the numbers are directly comparable with the paper's
0.59 accuracy / 0.52 macro-F1.

In [ ]:
for f in ['dev', 'paper_20_test', 'ours_20_test']:
    split = 'dev' if f == 'dev' else 'test'
    !python scripts/score_multivers.py results/classification/classifier/{f}.jsonl --split {split}

## Save the results

Downloads the predictions and the training log (small). The 440 MB model is copied to your Google
Drive only if you run the last cell.

In [ ]:
!cp checkpoints/classifier/config.json results/classification/classifier/
!cd results/classification && zip -qr /content/classifier_results.zip classifier
from google.colab import files
files.download('/content/classifier_results.zip')

In [ ]:
# Optional: keep the trained model in Google Drive.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/CiteAudit && cp -r checkpoints/classifier /content/drive/MyDrive/CiteAudit/